In [1]:
# =============================================================================
# Imports
# =============================================================================
import os

import numpy as np
import pandas as pd
import xarray as xr

# 1. Load Grid-Level HERA and ECAD Data

In [2]:
print(os.getcwd())

/work/ch0636/g300138/LowFlowHeatWave/CompoundLowFlowHeatWave


In [3]:
in_dir = 'Data/'

proto_lon = 6.15
proto_lat = 45.85

coord_tag = f'lon{proto_lon}_lat{proto_lat}'

q_threshold=10
tx_threshold=90

HERA_dis_df = pd.read_csv(f'{in_dir}HERA_dis_{coord_tag}.csv')
TxRaw_df = pd.read_csv(f'{in_dir}TxRaw_{coord_tag}.csv')
HERA_QXThreshold_df = pd.read_csv(f'{in_dir}HERA_Q{q_threshold}Threshold_{coord_tag}.csv')
ECAD_TXThreshold_df = pd.read_csv(f'{in_dir}ECAD_TX{tx_threshold}Threshold_{coord_tag}.csv')

# print(HERA_dis_df.head())
# print(TxRaw_df.head())
# print(HERA_QXThreshold_df.head())
# print(ECAD_TXThreshold_df.head())

# 2. Event Filtering Utilities

In [4]:
def filter_summer_low_flow_events(
    events_df,
    summer_start_months=[6, 7, 8, 9],
    summer_end_months=[6, 7, 8, 9, 10],
    min_duration=3
):
    """
    Filter low-flow events to those starting and ending within specified summer months,
    and meeting a minimum duration threshold.

    Parameters
    ----------
    events_df : DataFrame
        Output of detect_low_flow_events(), must contain 'start_date', 'start_month',
        'duration' columns.
    summer_start_months : list of int, optional
        Months (1-12) allowed for event start. Default [6,7,8,9] (JJAS).
    summer_end_months : list of int, optional
        Months (1-12) allowed for event end. Default [6,7,8,9,10] (JJASO).
    min_duration : int, optional
        Minimum event duration (days) to retain. Default 3.

    Returns
    -------
    filtered_df : DataFrame
        Subset of events_df meeting all criteria, with an added 'end_date' and
        'end_month' column (computed from start_date + duration) for transparency.
    """

    df = events_df.copy()

    # --- Compute end_date / end_month from start_date + duration ---
    start_dt = pd.to_datetime(df['start_date'])
    end_dt = start_dt + pd.to_timedelta(df['duration'] - 1, unit='D')  # inclusive end day
    df['end_date'] = end_dt.dt.strftime('%Y-%m-%d')
    df['end_month'] = end_dt.dt.month

    # --- Apply filters ---
    start_mask = df['start_month'].isin(summer_start_months)
    end_mask = df['end_month'].isin(summer_end_months)
    duration_mask = df['duration'] >= min_duration

    filtered_df = df[start_mask & end_mask & duration_mask].reset_index(drop=True)

    return filtered_df

# 3. Low-Flow Event Detection

In [5]:
def detect_low_flow_events(dis_df, threshold_df, start_date="1961-01-01", end_date="2020-12-31"):
    """
    Detect low-flow events at a single grid cell using a day-of-year climatological threshold.
    """

    # --- 1. Ensure datetime index and slice to the desired period ---
    dis = dis_df.copy()
    if 'time' in dis.columns:
        dis['time'] = pd.to_datetime(dis['time'])
        dis = dis.set_index('time')
    dis = dis.sort_index()
    dis = dis.loc[start_date:end_date]

    # --- 2. Map each timestamp's day-of-year to its threshold ---
    doy = dis.index.dayofyear.copy()
    doy = np.where(doy == 366, 365, doy)  # leap day -> use day 365 threshold

    doy_to_thresh = threshold_df.set_index('day')['Q10_threshold'].to_dict()
    threshold_series = pd.Series(doy, index=dis.index).map(doy_to_thresh)

    # --- 3. Identify discharge column ---
    dis_col = [c for c in dis.columns if c.lower() in ('dis', 'discharge')][0]
    discharge = dis[dis_col]

    # --- 4. Flag low-flow timesteps ---
    valid = discharge.notna() & threshold_series.notna()
    is_low_flow = (discharge < threshold_series) & valid
    is_low_flow_clean = is_low_flow.fillna(False).astype(int).values

    # --- 5. Find event starts/ends via discrete derivative ---
    diff = np.diff(is_low_flow_clean)
    starts = list(np.where(diff == 1)[0] + 1)
    ends = list(np.where(diff == -1)[0] + 1)

    if is_low_flow_clean[0] == 1:
        starts = [0] + starts
    if is_low_flow_clean[-1] == 1:
        ends = ends + [len(is_low_flow_clean)]

    # --- 6. Build event records ---
    dates = dis.index
    records = []
    for s, e in zip(starts, ends):
        event_discharge = discharge.iloc[s:e]
        event_threshold = threshold_series.iloc[s:e]
        start_dt = dates[s]

        records.append({
            'threshold': event_threshold.iloc[0],
            'year': start_dt.year,
            'start_date': start_dt.strftime('%Y-%m-%d'),
            'start_day': start_dt.day,
            'start_month': start_dt.month,
            'duration': e - s,
            'mean_discharge': event_discharge.mean(),
            'min_discharge': event_discharge.min(),
            'mean_threshold': event_threshold.mean(),
        })

    return pd.DataFrame(records)

In [6]:
low_flow_events = detect_low_flow_events(
    HERA_dis_df, HERA_QXThreshold_df,
    start_date="1961-01-01", end_date="2020-12-31"
)
low_flow_events.head()

,threshold,year,start_date,start_day,start_month,duration,mean_discharge,min_discharge,mean_threshold
0,5.540244,1961,1961-03-30,30,3,2,5.336426,5.186691,5.540244
1,2.835726,1961,1961-05-11,11,5,8,1.350758,0.856720,2.556730
2,0.851965,1961,1961-06-21,21,6,1,0.680618,0.680618,0.851965
3,0.617650,1961,1961-07-09,9,7,1,0.574997,0.574997,0.617650
4,0.804504,1961,1961-08-28,28,8,8,0.590466,0.490143,0.811084


In [7]:
summer_low_flow_events = filter_summer_low_flow_events(
    low_flow_events,
    min_duration=3
)
print(summer_low_flow_events.head())
print(f"Number of summer low-flow events: {len(summer_low_flow_events)}")

period1 = summer_low_flow_events[
    (summer_low_flow_events['year'] >= 1961) & (summer_low_flow_events['year'] <= 1990)
]
period2 = summer_low_flow_events[
    (summer_low_flow_events['year'] >= 1991) & (summer_low_flow_events['year'] <= 2020)
]

print(f"Number of summer low-flow events (1961-1990): {len(period1)}")
print(f"Number of summer low-flow events (1991-2020): {len(period2)}")

   threshold  year  start_date  start_day  start_month  duration  \
0   0.804504  1961  1961-08-28         28            8         8   
1   0.879111  1961  1961-09-12         12            9        18   
2   0.758704  1962  1962-06-24         24            6        11   
3   0.617650  1962  1962-07-09          9            7         5   
4   0.604799  1962  1962-07-25         25            7         3   

   mean_discharge  min_discharge  mean_threshold    end_date  end_month  
0        0.590466       0.490143        0.811084  1961-09-04          9  
1        0.743621       0.648743        0.936865  1961-09-29          9  
2        0.527095       0.401672        0.700101  1962-07-04          7  
3        0.461487       0.359375        0.606979  1962-07-13          7  
4        0.393468       0.345215        0.610318  1962-07-27          7  
Number of summer low-flow events: 102
Number of summer low-flow events (1961-1990): 33
Number of summer low-flow events (1991-2020): 69


# 4. Heatwave Event Detection

In [8]:
def detect_heatwave_events(tx_df, threshold_df, start_date="1961-01-01", end_date="2020-12-31"):
    """
    Detect heatwave events at a single grid cell using a day-of-year climatological threshold.

    Parameters
    ----------
    tx_df : DataFrame with 'time', 'lon', 'lat', 'tx' columns
    threshold_df : DataFrame with 'day', 'lon', 'lat', 'TX{x}_threshold' columns
    start_date, end_date : str, slicing bounds (inclusive)

    Returns
    -------
    events_df : DataFrame, one row per detected heatwave event
    """

    # --- 1. Ensure datetime index and slice to the desired period ---
    tx = tx_df.copy()
    if 'time' in tx.columns:
        tx['time'] = pd.to_datetime(tx['time'])
        tx = tx.set_index('time')
    tx = tx.sort_index()
    tx = tx.loc[start_date:end_date]

    # --- 2. Grab lon/lat for output (assumed constant, single grid cell) ---
    lon_val = round(tx['lon'].iloc[0], 2)
    lat_val = round(tx['lat'].iloc[0], 2)

    # --- 3. Map each timestamp's day-of-year to its threshold ---
    doy = tx.index.dayofyear.copy()
    doy = np.where(doy == 366, 365, doy)  # leap day -> use day 365 threshold

    thresh_col = [c for c in threshold_df.columns if 'threshold' in c.lower()][0]
    doy_to_thresh = threshold_df.set_index('day')[thresh_col].to_dict()
    threshold_series = pd.Series(doy, index=tx.index).map(doy_to_thresh)

    # --- 4. Identify temperature column ---
    tx_col = [c for c in tx.columns if c.lower() in ('tx', 'tmax', 'temperature')][0]
    temperature = tx[tx_col]

    # --- 5. Flag extreme (heatwave) timesteps ---
    valid = temperature.notna() & threshold_series.notna()
    is_extreme = (temperature > threshold_series) & valid
    is_extreme_clean = is_extreme.fillna(False).astype(int).values

    # --- 6. Find event starts/ends via discrete derivative ---
    diff = np.diff(is_extreme_clean)
    starts = list(np.where(diff == 1)[0] + 1)
    ends = list(np.where(diff == -1)[0] + 1)

    if is_extreme_clean[0] == 1:
        starts = [0] + starts
    if is_extreme_clean[-1] == 1:
        ends = ends + [len(is_extreme_clean)]

    # --- 7. Build event records, discarding any window with NaNs ---
    dates = tx.index
    records = []
    for s, e in zip(starts, ends):
        event_temp = temperature.iloc[s:e]
        event_thresh = threshold_series.iloc[s:e]

        # NaN Check: discard event if any NaN in temp or threshold over the window
        if event_temp.isna().any() or event_thresh.isna().any():
            continue

        start_dt = dates[s]
        records.append({
            'lon': lon_val,
            'lat': lat_val,
            'threshold': event_thresh.iloc[0],
            'year': start_dt.year,
            'start_date': start_dt.strftime('%Y-%m-%d'),
            'start_day': start_dt.day,
            'start_month': start_dt.month,
            'duration': e - s,
            'mean_temp': event_temp.mean(),
            'max_temp': event_temp.max(),
            'mean_threshold': event_thresh.mean(),
        })

    return pd.DataFrame(records)

In [9]:
heatwave_events = detect_heatwave_events(
    TxRaw_df, ECAD_TXThreshold_df,
    start_date="1961-01-01", end_date="2020-12-31"
)
heatwave_events.head()

summer_heatwave_events = filter_summer_low_flow_events(
    heatwave_events,
    summer_start_months=[6, 7, 8, 9],
    summer_end_months=[6, 7, 8, 9, 10],
    min_duration=3
)

print(summer_heatwave_events.head())
print(f"Number of summer heatwave events: {len(summer_heatwave_events)}")

period1 = summer_heatwave_events[
    (summer_heatwave_events['year'] >= 1961) & (summer_heatwave_events['year'] <= 1990)
]
period2 = summer_heatwave_events[
    (summer_heatwave_events['year'] >= 1991) & (summer_heatwave_events['year'] <= 2020)
]

print(f"Number of summer heatwave events (1961-1990): {len(period1)}")
print(f"Number of summer heatwave events (1991-2020): {len(period2)}")

    lon    lat  threshold  year  start_date  start_day  start_month  duration  \
0  6.15  45.85  28.072001  1961  1961-08-29         29            8         5   
1  6.15  45.85  25.834000  1961  1961-09-16         16            9         5   
2  6.15  45.85  24.463999  1961  1961-09-23         23            9         7   
3  6.15  45.85  29.873999  1962  1962-08-12         12            8         3   
4  6.15  45.85  30.411999  1964  1964-07-16         16            7         6   

   mean_temp   max_temp  mean_threshold    end_date  end_month  
0  28.212000  28.580000       27.634400  1961-09-02          9  
1  26.995999  27.609999       25.525599  1961-09-20          9  
2  25.068571  26.380000       23.771428  1961-09-29          9  
3  30.493333  31.230000       29.814332  1962-08-14          8  
4  32.363332  33.899998       30.490166  1964-07-21          7  
Number of summer heatwave events: 93
Number of summer heatwave events (1961-1990): 30
Number of summer heatwave events (199

# 5. Compound Event Detection

In [10]:
def detect_compound_events(
    low_flow_df, heatwave_df,
    dis_df, tx_df,
    dis_threshold_df, tx_threshold_df,
    min_overlap_days=1
):
    """
    Detect compound low-flow + heatwave events via cross-join interval overlap.

    Parameters
    ----------
    low_flow_df : DataFrame of summer low-flow events (needs start_date, duration)
    heatwave_df : DataFrame of summer heatwave events (needs start_date, duration)
    dis_df : raw discharge DataFrame (time, dis) — for flow deficit enrichment
    tx_df : raw max temperature DataFrame (time, tx) — for temp excess enrichment
    dis_threshold_df : discharge threshold DataFrame (day, Q{x}_threshold)
    tx_threshold_df : temperature threshold DataFrame (day, TX{x}_threshold)
    min_overlap_days : int, minimum overlap (days) to qualify as compound. Default 1.

    Returns
    -------
    compound_df : DataFrame, one row per compound event
    """

    if len(low_flow_df) == 0 or len(heatwave_df) == 0:
        return pd.DataFrame()

    # --- Prep discharge/temp lookups for enrichment ---
    dis = dis_df.copy()
    if 'time' in dis.columns:
        dis['time'] = pd.to_datetime(dis['time'])
        dis = dis.set_index('time').sort_index()
    dis_col = [c for c in dis.columns if c.lower() in ('dis', 'discharge')][0]

    tx = tx_df.copy()
    if 'time' in tx.columns:
        tx['time'] = pd.to_datetime(tx['time'])
        tx = tx.set_index('time').sort_index()
    tx_col = [c for c in tx.columns if c.lower() in ('tx', 'tmax', 'temperature')][0]

    dis_thresh_col = [c for c in dis_threshold_df.columns if 'threshold' in c.lower()][0]
    tx_thresh_col = [c for c in tx_threshold_df.columns if 'threshold' in c.lower()][0]
    doy_to_dis_thresh = dis_threshold_df.set_index('day')[dis_thresh_col].to_dict()
    doy_to_tx_thresh = tx_threshold_df.set_index('day')[tx_thresh_col].to_dict()

    # --- Compute end dates for each event set ---
    lf = low_flow_df.copy()
    lf['start_date'] = pd.to_datetime(lf['start_date'])
    lf['end_date'] = lf['start_date'] + pd.to_timedelta(lf['duration'] - 1, unit='D')

    hw = heatwave_df.copy()
    hw['start_date'] = pd.to_datetime(hw['start_date'])
    hw['end_date'] = hw['start_date'] + pd.to_timedelta(hw['duration'] - 1, unit='D')

    # --- Cartesian cross-join ---
    lf['_key'] = 1
    hw['_key'] = 1
    cross = lf.merge(hw, on='_key', suffixes=('_LF', '_HW')).drop(columns='_key')

    # --- Compute overlap window ---
    cross['overlap_start'] = np.maximum(cross['start_date_LF'], cross['start_date_HW'])
    cross['overlap_end'] = np.minimum(cross['end_date_LF'], cross['end_date_HW'])
    cross['overlap_days'] = (cross['overlap_end'] - cross['overlap_start']).dt.days + 1
    cross['overlap_days'] = cross['overlap_days'].clip(lower=0)

    # --- Filter to qualifying compound events ---
    compound = cross[cross['overlap_days'] >= min_overlap_days].reset_index(drop=True)

    if len(compound) == 0:
        return pd.DataFrame()

    # --- Hazard precedence & lag ---
    def get_first_event(row):
        if row['start_date_LF'] < row['start_date_HW']:
            return 'LowFlow'
        elif row['start_date_HW'] < row['start_date_LF']:
            return 'HeatWave'
        else:
            return 'Simultaneous'

    compound['first_event'] = compound.apply(get_first_event, axis=1)
    compound['time_difference_days'] = (
        compound['start_date_HW'] - compound['start_date_LF']
    ).dt.days.abs()

    # --- Physical intensity enrichment over overlap window ---
    records = []
    for _, row in compound.iterrows():
        overlap_dates = pd.date_range(row['overlap_start'], row['overlap_end'], freq='D')

        # Temperature excess
        temp_vals = tx.reindex(overlap_dates)[tx_col]
        doy_tx = np.where(overlap_dates.dayofyear == 366, 365, overlap_dates.dayofyear)
        tx_thresh_vals = pd.Series([doy_to_tx_thresh.get(d, np.nan) for d in doy_tx], index=overlap_dates)
        temp_excess = temp_vals - tx_thresh_vals
        mean_temp_excess = temp_excess.mean()
        max_temp_excess = temp_excess.max()

        # Flow deficit with negative-threshold guard
        dis_vals = dis.reindex(overlap_dates)[dis_col]
        doy_dis = np.where(overlap_dates.dayofyear == 366, 365, overlap_dates.dayofyear)
        dis_thresh_vals = pd.Series([doy_to_dis_thresh.get(d, np.nan) for d in doy_dis], index=overlap_dates)

        if (dis_thresh_vals < 0).any():
            flow_deficit_flag = 'negative_threshold'
            mean_flow_deficit = np.nan
            max_flow_deficit = np.nan
        else:
            flow_deficit_flag = 'ok'
            flow_deficit = dis_thresh_vals - dis_vals
            mean_flow_deficit = flow_deficit.mean()
            max_flow_deficit = flow_deficit.max()

        records.append({
            'lon': row.get('lon_HW', row.get('lon', np.nan)),
            'lat': row.get('lat_HW', row.get('lat', np.nan)),
            'start_date_LF': row['start_date_LF'].strftime('%Y-%m-%d'),
            'end_date_LF': row['end_date_LF'].strftime('%Y-%m-%d'),
            'duration_LF': row['duration_LF'],
            'start_date_HW': row['start_date_HW'].strftime('%Y-%m-%d'),
            'end_date_HW': row['end_date_HW'].strftime('%Y-%m-%d'),
            'duration_HW': row['duration_HW'],
            'overlap_start': row['overlap_start'].strftime('%Y-%m-%d'),
            'overlap_end': row['overlap_end'].strftime('%Y-%m-%d'),
            'overlap_days': row['overlap_days'],
            'first_event': row['first_event'],
            'time_difference_days': row['time_difference_days'],
            'actual_mean_temperature_excess': mean_temp_excess,
            'actual_max_temperature_excess': max_temp_excess,
            'flow_deficit_status': flow_deficit_flag,
            'actual_mean_flow_deficit': mean_flow_deficit,
            'actual_max_flow_deficit': max_flow_deficit,
        })

    return pd.DataFrame(records)

In [11]:
compound_events = detect_compound_events(
    summer_low_flow_events, summer_heatwave_events,
    HERA_dis_df, TxRaw_df,
    HERA_QXThreshold_df, ECAD_TXThreshold_df,
    min_overlap_days=1
)

print(compound_events.head())
print(f"Number of compound events: {len(compound_events)}")

compound_events['start_date_LF'] = pd.to_datetime(compound_events['start_date_LF'])
compound_events['year'] = compound_events['start_date_LF'].dt.year

period1 = compound_events[
    (compound_events['year'] >= 1961) & (compound_events['year'] <= 1990)
]
period2 = compound_events[
    (compound_events['year'] >= 1991) & (compound_events['year'] <= 2020)
]

print(f"Number of compound events (1961-1990): {len(period1)}")
print(f"Number of compound events (1991-2020): {len(period2)}")

    lon    lat start_date_LF end_date_LF  duration_LF start_date_HW  \
0  6.15  45.85    1961-08-28  1961-09-04            8    1961-08-29   
1  6.15  45.85    1961-09-12  1961-09-29           18    1961-09-16   
2  6.15  45.85    1961-09-12  1961-09-29           18    1961-09-23   
3  6.15  45.85    1964-07-18  1964-07-22            5    1964-07-16   
4  6.15  45.85    1976-06-09  1976-06-26           18    1976-06-13   

  end_date_HW  duration_HW overlap_start overlap_end  overlap_days  \
0  1961-09-02            5    1961-08-29  1961-09-02             5   
1  1961-09-20            5    1961-09-16  1961-09-20             5   
2  1961-09-29            7    1961-09-23  1961-09-29             7   
3  1964-07-21            6    1964-07-18  1964-07-21             4   
4  1976-06-15            3    1976-06-13  1976-06-15             3   

  first_event  time_difference_days  actual_mean_temperature_excess  \
0     LowFlow                     1                        0.577600   
1     LowF

# 6. Compound Event Statistics

## 6.1 Mean Annual Event Count, Duration, and Intensity

In [13]:
def compute_compound_stats(compound_events, start_year, end_year, N_years=30, verbose=True):
    """
    Compute mean annual frequency, duration, and intensity statistics
    for a compound event catalog over a fixed climatological baseline.

    Duration and intensity are computed as: for each year, take the mean
    across that year's events (0 if no events that year), then average
    those per-year values over N_years.
    """
    df = compound_events.copy()
    df['overlap_start'] = pd.to_datetime(df['overlap_start'])
    df['year'] = df['overlap_start'].dt.year

    period_df = df[(df['year'] >= start_year) & (df['year'] <= end_year)]
    K_g = len(period_df)

    if verbose:
        print("=" * 65)
        print(f"PERIOD: {start_year}-{end_year}  (N_years = {N_years})")
        print("=" * 65)

    if K_g == 0:
        if verbose:
            print("No events in this period.")
            print()
        return {
            'period': f'{start_year}-{end_year}',
            'frequency': 0.0,
            'annual_duration': 0.0,
            'annual_temp_excess': 0.0,
            'annual_flow_deficit': 0.0,
        }

    all_years = list(range(start_year, end_year + 1))

    # --- Frequency (total events / N_years) ---
    frequency = K_g / N_years
    if verbose:
        print(f"Number of events (K_g)              : {K_g}")
        print(f"Frequency = K_g / N_years            = {K_g} / {N_years} = {frequency:.6f}")
        print()

    # --- Build one combined per-year table ---
    grouped_duration = period_df.groupby('year')['overlap_days'].apply(list)
    grouped_temp = period_df.groupby('year')['actual_mean_temperature_excess'].apply(list)
    grouped_flow = period_df.groupby('year')['actual_mean_flow_deficit'].apply(list)

    rows = []
    for yr in all_years:
        dur_vals = grouped_duration.get(yr, [])
        temp_vals = grouped_temp.get(yr, [])
        flow_vals = grouped_flow.get(yr, [])
        n = len(dur_vals)

        rows.append({
            'year': yr,
            'n_events': n,
            'durations': dur_vals,
            'annual_mean_event_duration': float(np.mean(dur_vals)) if n > 0 else 0.0,
            'temp_excess_values': temp_vals,
            'annual_mean_temp_excess': float(np.mean(temp_vals)) if n > 0 else 0.0,
            'flow_deficit_values': flow_vals,
            'annual_mean_flow_deficit': float(np.mean(flow_vals)) if n > 0 else 0.0,
        })

    year_table = pd.DataFrame(rows)

    # --- Mean annual aggregates: sum of per-year means / N_years ---
    mean_annual_duration = year_table['annual_mean_event_duration'].sum() / N_years
    mean_annual_temp_excess = year_table['annual_mean_temp_excess'].sum() / N_years
    mean_annual_flow_deficit = year_table['annual_mean_flow_deficit'].sum() / N_years

    if verbose:
        print("Per-year breakdown:")
        print(year_table.to_string(index=False))
        print()
        print(f"Sum of annual mean event durations       : {year_table['annual_mean_event_duration'].sum():.6f}")
        print(f"Mean annual event duration = sum/N_years  = {year_table['annual_mean_event_duration'].sum():.6f} / {N_years} = {mean_annual_duration:.6f}")
        print()
        print(f"Sum of annual mean temp excess            : {year_table['annual_mean_temp_excess'].sum():.6f}")
        print(f"Mean annual temp excess = sum/N_years      = {year_table['annual_mean_temp_excess'].sum():.6f} / {N_years} = {mean_annual_temp_excess:.6f}")
        print()
        print(f"Sum of annual mean flow deficit            : {year_table['annual_mean_flow_deficit'].sum():.6f}")
        print(f"Mean annual flow deficit = sum/N_years      = {year_table['annual_mean_flow_deficit'].sum():.6f} / {N_years} = {mean_annual_flow_deficit:.6f}")
        print()

    return {
        'period': f'{start_year}-{end_year}',
        'frequency': frequency,
        'annual_duration': mean_annual_duration,
        'annual_temp_excess': mean_annual_temp_excess,
        'annual_flow_deficit': mean_annual_flow_deficit,
    }


# --- Compute for both baselines ---
stats_1961_1990 = compute_compound_stats(compound_events, 1961, 1990, N_years=30, verbose=False)
stats_1991_2020 = compute_compound_stats(compound_events, 1991, 2020, N_years=30, verbose=False)

stats_df = pd.DataFrame([stats_1961_1990, stats_1991_2020]).set_index('period')

# --- Relative delta: (1991-2020 minus 1961-1990) / 1961-1990 ---
v1961 = stats_df.loc['1961-1990']
v1991 = stats_df.loc['1991-2020']
relative_delta = (v1991 - v1961) / v1961.replace(0, np.nan)
stats_df.loc['relative_delta'] = relative_delta

stats_df

,frequency,annual_duration,annual_temp_excess,annual_flow_deficit
period,,,,
1961-1990,0.333333,1.455556,0.456083,0.048305
1991-2020,1.366667,2.901111,1.528489,0.138902
relative_delta,3.100000,0.993130,2.351339,1.875506


## 6.2 LMDI Decomposition

In [14]:
# =====================================================================
# 1. Ensure datetime columns exist
# =====================================================================
compound_events['overlap_start'] = pd.to_datetime(compound_events['overlap_start'])
summer_low_flow_events['start_date'] = pd.to_datetime(summer_low_flow_events['start_date'])
summer_heatwave_events['start_date'] = pd.to_datetime(summer_heatwave_events['start_date'])

# =====================================================================
# 2. Split each catalog into the two periods
# =====================================================================
Period_1961_CE_df = compound_events[
    (compound_events['overlap_start'].dt.year >= 1961) & (compound_events['overlap_start'].dt.year <= 1990)
]
Period_1991_CE_df = compound_events[
    (compound_events['overlap_start'].dt.year >= 1991) & (compound_events['overlap_start'].dt.year <= 2020)
]

Period_1961_LFE_df = summer_low_flow_events[
    (summer_low_flow_events['start_date'].dt.year >= 1961) & (summer_low_flow_events['start_date'].dt.year <= 1990)
]
Period_1991_LFE_df = summer_low_flow_events[
    (summer_low_flow_events['start_date'].dt.year >= 1991) & (summer_low_flow_events['start_date'].dt.year <= 2020)
]

Period_1961_HWE_df = summer_heatwave_events[
    (summer_heatwave_events['start_date'].dt.year >= 1961) & (summer_heatwave_events['start_date'].dt.year <= 1990)
]
Period_1991_HWE_df = summer_heatwave_events[
    (summer_heatwave_events['start_date'].dt.year >= 1991) & (summer_heatwave_events['start_date'].dt.year <= 2020)
]

# =====================================================================
# 3. Event counts
# =====================================================================
n_CEs_Period_1991 = len(Period_1991_CE_df)
n_CEs_Period_1961 = len(Period_1961_CE_df)
n_LFEs_Period_1991 = len(Period_1991_LFE_df)
n_LFEs_Period_1961 = len(Period_1961_LFE_df)
n_HWEs_Period_1991 = len(Period_1991_HWE_df)
n_HWEs_Period_1961 = len(Period_1961_HWE_df)

# =====================================================================
# 4. Guard: need at least one LFE in both periods
# =====================================================================
if n_LFEs_Period_1991 == 0 or n_LFEs_Period_1961 == 0:
    raise ValueError(
        f"Cannot run LMDI decomposition: zero LFEs in one or both periods "
        f"(1961: {n_LFEs_Period_1961}, 1991: {n_LFEs_Period_1991}). "
        f"LMDI requires strictly positive N, D, R in both periods (log undefined at 0)."
    )

# =====================================================================
# 5. Total LFE duration (days)
# =====================================================================
total_LFE_days_1991 = Period_1991_LFE_df['duration'].sum()
total_LFE_days_1961 = Period_1961_LFE_df['duration'].sum()

if total_LFE_days_1991 == 0 or total_LFE_days_1961 == 0:
    raise ValueError("Total LFE duration is zero in one or both periods — cannot compute mean duration or HWE rate.")

# =====================================================================
# 6. Mean LFE duration (days/event)
# =====================================================================
mean_LFE_duration_1991 = total_LFE_days_1991 / n_LFEs_Period_1991
mean_LFE_duration_1961 = total_LFE_days_1961 / n_LFEs_Period_1961

# =====================================================================
# 7. HWE rate during LFEs = CEs / total LFE duration
# =====================================================================
R_1991 = n_CEs_Period_1991 / total_LFE_days_1991
R_1961 = n_CEs_Period_1961 / total_LFE_days_1961

# =====================================================================
# 8. Assign the three LMDI drivers
# =====================================================================
N1, D1, R1 = n_LFEs_Period_1961, mean_LFE_duration_1961, R_1961
N2, D2, R2 = n_LFEs_Period_1991, mean_LFE_duration_1991, R_1991

# Reconstruction check (silent unless it fails)
CE1_check = N1 * D1 * R1
CE2_check = N2 * D2 * R2
if not np.isclose(CE1_check, n_CEs_Period_1961, rtol=1e-6) or not np.isclose(CE2_check, n_CEs_Period_1991, rtol=1e-6):
    print(f"⚠ WARNING: N*D*R does not reconstruct CE count exactly "
          f"(N1*D1*R1={CE1_check:.4f} vs actual={n_CEs_Period_1961}, "
          f"N2*D2*R2={CE2_check:.4f} vs actual={n_CEs_Period_1991})")

# =====================================================================
# 9. LMDI decomposition functions
# =====================================================================

def logarithmic_decomposition(N1, D1, R1, N2, D2, R2):
    """
    Fallback for LMDI when CE1 or CE2 is zero (standard log-mean weight
    is undefined). Uses the arithmetic-mean-weight variant instead.
    """
    CE1 = N1 * D1 * R1
    CE2 = N2 * D2 * R2

    L = (CE1 + CE2) / 2

    def safe_log_ratio(a, b):
        if a <= 0 or b <= 0:
            return 0.0
        return np.log(a / b)

    contrib_N = L * safe_log_ratio(N2, N1)
    contrib_D = L * safe_log_ratio(D2, D1)
    contrib_R = L * safe_log_ratio(R2, R1)

    total_change = CE2 - CE1
    sum_contributions = contrib_N + contrib_D + contrib_R
    residual = total_change - sum_contributions

    pct_N = contrib_N / total_change * 100 if total_change != 0 else 0
    pct_D = contrib_D / total_change * 100 if total_change != 0 else 0
    pct_R = contrib_R / total_change * 100 if total_change != 0 else 0

    return {
        'absolute': {'N': contrib_N, 'D': contrib_D, 'R': contrib_R},
        'percentage': {'N': pct_N, 'D': pct_D, 'R': pct_R},
        'total_change': total_change,
        'sum_contributions': sum_contributions,
        'residual': residual,
        'method': 'arithmetic-mean fallback (CE1 or CE2 was zero)'
    }


def lmdi_decomposition(N1, D1, R1, N2, D2, R2):
    """
    LMDI decomposition for perfect (additive, residual-free) attribution.
    Uses the logarithmic mean weight with natural log, standard in
    IPCC/climate-attribution LMDI applications.
    """
    for name, val in [('N1', N1), ('D1', D1), ('R1', R1), ('N2', N2), ('D2', D2), ('R2', R2)]:
        if val < 0:
            raise ValueError(f"{name} = {val} is negative — LMDI requires non-negative driver values.")

    CE1 = N1 * D1 * R1
    CE2 = N2 * D2 * R2

    if CE1 == 0 or CE2 == 0:
        return logarithmic_decomposition(N1, D1, R1, N2, D2, R2)

    def log_mean(a, b):
        if a <= 0 or b <= 0:
            raise ValueError("log_mean requires strictly positive inputs.")
        if abs(a - b) < 1e-10:
            return a
        return (a - b) / np.log(a / b)

    L = log_mean(CE1, CE2)

    def safe_log_ratio(a, b, label):
        if a <= 0 or b <= 0:
            raise ValueError(f"Cannot compute log ratio for {label}: values must be > 0 (got {a}, {b}).")
        return np.log(a / b)

    contrib_N = L * safe_log_ratio(N2, N1, 'N')
    contrib_D = L * safe_log_ratio(D2, D1, 'D')
    contrib_R = L * safe_log_ratio(R2, R1, 'R')

    total_change = CE2 - CE1
    sum_contributions = contrib_N + contrib_D + contrib_R
    residual = total_change - sum_contributions

    pct_N = contrib_N / total_change * 100 if total_change != 0 else 0
    pct_D = contrib_D / total_change * 100 if total_change != 0 else 0
    pct_R = contrib_R / total_change * 100 if total_change != 0 else 0

    return {
        'absolute': {'N': contrib_N, 'D': contrib_D, 'R': contrib_R},
        'percentage': {'N': pct_N, 'D': pct_D, 'R': pct_R},
        'total_change': total_change,
        'sum_contributions': sum_contributions,
        'residual': residual,
        'method': 'logarithmic mean (standard LMDI)'
    }


# =====================================================================
# 10. Run decomposition
# =====================================================================
result = lmdi_decomposition(N1, D1, R1, N2, D2, R2)

# =====================================================================
# 11. Print results
# =====================================================================
print(f"N 1961 period: {N1}")
print(f"D 1961 period: {D1:.6f}")
print(f"R 1961 period: {R1:.6f}")
print()
print(f"N 1991 period: {N2}")
print(f"D 1991 period: {D2:.6f}")
print(f"R 1991 period: {R2:.6f}")
print()

print("LMDI DECOMPOSITION")
print(f"N: {result['absolute']['N']:6.2f} CEs ({result['percentage']['N']:5.1f}%)")
print(f"D: {result['absolute']['D']:6.2f} CEs ({result['percentage']['D']:5.1f}%)")
print(f"R: {result['absolute']['R']:6.2f} CEs ({result['percentage']['R']:5.1f}%)")

N 1961 period: 33
D 1961 period: 6.727273
R 1961 period: 0.045045

N 1991 period: 69
D 1991 period: 6.188406
R 1991 period: 0.096019

LMDI DECOMPOSITION
N:  16.21 CEs ( 52.3%)
D:  -1.83 CEs ( -5.9%)
R:  16.63 CEs ( 53.6%)


## 6.3 Conditional Risk: Heatwave Given Low Flow, and Low Flow Given Heatwave

In [20]:
def compute_day_based_ce_ratios(compound_events, summer_low_flow_events, summer_heatwave_events,
                                   start_year=1991, end_year=2020, verbose=True):
    """
    Compute day-based compound event ratios:
      Heatwave risk given low flow = total compound-overlap days / total low-flow event days
      Low-flow risk given a heatwave = total compound-overlap days / total heatwave event days

    Uses your existing column names:
      compound_events: 'overlap_start' (datetime), 'overlap_days'
      summer_low_flow_events: 'start_date', 'duration'
      summer_heatwave_events: 'start_date', 'duration'
    """
    N_years = end_year - start_year + 1
    all_years = np.arange(start_year, end_year + 1)

    ce = compound_events.copy()
    ce['overlap_start'] = pd.to_datetime(ce['overlap_start'])
    ce['year'] = ce['overlap_start'].dt.year
    ce = ce[(ce['year'] >= start_year) & (ce['year'] <= end_year)]

    lf = summer_low_flow_events.copy()
    lf['start_date'] = pd.to_datetime(lf['start_date'])
    lf['year'] = lf['start_date'].dt.year
    lf = lf[(lf['year'] >= start_year) & (lf['year'] <= end_year)]

    hw = summer_heatwave_events.copy()
    hw['start_date'] = pd.to_datetime(hw['start_date'])
    hw['year'] = hw['start_date'].dt.year
    hw = hw[(hw['year'] >= start_year) & (hw['year'] <= end_year)]

    # --- Build annual day totals ---
    annual_df = pd.DataFrame({'year': all_years})

    ce_days_annual = ce.groupby('year')['overlap_days'].sum().reset_index(name='ce_days')
    lf_days_annual = lf.groupby('year')['duration'].sum().reset_index(name='lf_days')
    hw_days_annual = hw.groupby('year')['duration'].sum().reset_index(name='hw_days')

    annual_df = annual_df.merge(ce_days_annual, on='year', how='left')
    annual_df = annual_df.merge(lf_days_annual, on='year', how='left')
    annual_df = annual_df.merge(hw_days_annual, on='year', how='left')
    annual_df = annual_df.fillna(0)

    # --- Day-based ratios per year ---
    annual_df['heatwave_risk_given_lowflow'] = np.where(
        annual_df['lf_days'] > 0, annual_df['ce_days'] / annual_df['lf_days'], 0.0
    )
    annual_df['lowflow_risk_given_heatwave'] = np.where(
        annual_df['hw_days'] > 0, annual_df['ce_days'] / annual_df['hw_days'], 0.0
    )

    if verbose:
        print(f"{'-'*70}")
        print(f"ANNUAL DAY TOTALS AND RATIOS ({start_year}-{end_year})")
        print(f"{'-'*70}")
        print(annual_df.to_string(index=False))

    # --- Overall totals (sum across all years) ---
    total_ce_days = annual_df['ce_days'].sum()
    total_lf_days = annual_df['lf_days'].sum()
    total_hw_days = annual_df['hw_days'].sum()

    heatwave_risk_given_lowflow = total_ce_days / total_lf_days if total_lf_days > 0 else np.nan
    lowflow_risk_given_heatwave = total_ce_days / total_hw_days if total_hw_days > 0 else np.nan

    if verbose:
        print(f"\n{'='*70}")
        print(f"OVERALL DAY-BASED RATIOS (SUM ACROSS {N_years} YEARS)")
        print(f"{'='*70}")
        print(f"Total CE days: {total_ce_days:.0f}")
        print(f"Total LF days: {total_lf_days:.0f}")
        print(f"Total HW days: {total_hw_days:.0f}")
        print()

    print(f"Heatwave risk given low flow = {total_ce_days:.0f} / {total_lf_days:.0f} = "
          f"{heatwave_risk_given_lowflow:.4f}")
    print(f"Low-flow risk given a heatwave = {total_ce_days:.0f} / {total_hw_days:.0f} = "
          f"{lowflow_risk_given_heatwave:.4f}")

    # --- Mean annual ratio (average of yearly ratios, not overall) ---
    mean_annual_heatwave_risk = annual_df['heatwave_risk_given_lowflow'].mean()
    mean_annual_lowflow_risk = annual_df['lowflow_risk_given_heatwave'].mean()

    if verbose:
        print(f"\n{'='*70}")
        print(f"MEAN ANNUAL DAY-BASED RATIOS (AVERAGE OF {N_years} YEARLY RATIOS)")
        print(f"{'='*70}")
        print(f"Mean annual heatwave risk given low flow: {mean_annual_heatwave_risk:.4f}")
        print(f"Mean annual low-flow risk given a heatwave: {mean_annual_lowflow_risk:.4f}")

    return {
        'annual_df': annual_df,
        'heatwave_risk_given_lowflow': heatwave_risk_given_lowflow,               # pooled (sum/sum)
        'lowflow_risk_given_heatwave': lowflow_risk_given_heatwave,               # pooled (sum/sum)
        'mean_annual_heatwave_risk_given_lowflow': mean_annual_heatwave_risk,     # Method A: mean of annual ratios
        'mean_annual_lowflow_risk_given_heatwave': mean_annual_lowflow_risk,      # Method A: mean of annual ratios
    }


# --- Run for both periods ---
results_1991_2020 = compute_day_based_ce_ratios(
    compound_events, summer_low_flow_events, summer_heatwave_events,
    start_year=1991, end_year=2020, verbose=True
)

results_1961_1990 = compute_day_based_ce_ratios(
    compound_events, summer_low_flow_events, summer_heatwave_events,
    start_year=1961, end_year=1990, verbose=False
)

# --- Assemble into a summary table ---
# NOTE: pulling from the 'mean_annual_*' keys, since Method A (mean of annual ratios)
# is the quantity reported in the manuscript, not the pooled sum-of-days ratio.
risk_df = pd.DataFrame({
    '1961-1990': {
        'heatwave_risk_given_lowflow': results_1961_1990['mean_annual_heatwave_risk_given_lowflow'],
        'lowflow_risk_given_heatwave': results_1961_1990['mean_annual_lowflow_risk_given_heatwave'],
    },
    '1991-2020': {
        'heatwave_risk_given_lowflow': results_1991_2020['mean_annual_heatwave_risk_given_lowflow'],
        'lowflow_risk_given_heatwave': results_1991_2020['mean_annual_lowflow_risk_given_heatwave'],
    },
}).T  # rows = periods, columns = metrics

# --- Relative delta: (1991-2020 minus 1961-1990) / 1961-1990 ---
v1961 = risk_df.loc['1961-1990']
v1991 = risk_df.loc['1991-2020']
relative_delta = (v1991 - v1961) / v1961.replace(0, np.nan)
risk_df.loc['relative_delta'] = relative_delta

risk_df

----------------------------------------------------------------------
ANNUAL DAY TOTALS AND RATIOS (1991-2020)
----------------------------------------------------------------------
 year  ce_days  lf_days  hw_days  heatwave_risk_given_lowflow  lowflow_risk_given_heatwave
 1991      4.0     14.0     10.0                     0.285714                     0.400000
 1992      3.0     10.0     10.0                     0.300000                     0.300000
 1993      0.0      7.0      4.0                     0.000000                     0.000000
 1994      2.0      3.0      8.0                     0.666667                     0.250000
 1995      0.0      0.0      0.0                     0.000000                     0.000000
 1996      0.0      5.0      3.0                     0.000000                     0.000000
 1997      3.0     14.0      3.0                     0.214286                     1.000000
 1998      2.0      6.0      5.0                     0.333333                     0.40000

,heatwave_risk_given_lowflow,lowflow_risk_given_heatwave
1961-1990,0.148772,0.188283
1991-2020,0.375345,0.453862
relative_delta,1.522954,1.410534


### Revision 2 Reviwer 1 comment

In [16]:
def compute_day_based_ce_ratios(compound_events, summer_low_flow_events, summer_heatwave_events,
                                   start_year=1991, end_year=2020, verbose=True):
    """
    Compute day-based compound event ratios:
      Heatwave risk given low flow = total compound-overlap days / total low-flow event days
      Low-flow risk given a heatwave = total compound-overlap days / total heatwave event days

    Uses your existing column names:
      compound_events: 'overlap_start' (datetime), 'overlap_days'
      summer_low_flow_events: 'start_date', 'duration'
      summer_heatwave_events: 'start_date', 'duration'

    Zero-day handling (per Reviewer 1, second revision):
      - Conditioning event occurred (lf_days > 0 or hw_days > 0) but no compound overlap:
        ratio is well-defined and equals 0.0. Included in the mean.
      - Conditioning event did NOT occur that year (lf_days == 0 or hw_days == 0):
        ratio is undefined (0/0), set to NaN, and EXCLUDED from the mean via nanmean.
    """
    N_years = end_year - start_year + 1
    all_years = np.arange(start_year, end_year + 1)

    ce = compound_events.copy()
    ce['overlap_start'] = pd.to_datetime(ce['overlap_start'])
    ce['year'] = ce['overlap_start'].dt.year
    ce = ce[(ce['year'] >= start_year) & (ce['year'] <= end_year)]

    lf = summer_low_flow_events.copy()
    lf['start_date'] = pd.to_datetime(lf['start_date'])
    lf['year'] = lf['start_date'].dt.year
    lf = lf[(lf['year'] >= start_year) & (lf['year'] <= end_year)]

    hw = summer_heatwave_events.copy()
    hw['start_date'] = pd.to_datetime(hw['start_date'])
    hw['year'] = hw['start_date'].dt.year
    hw = hw[(hw['year'] >= start_year) & (hw['year'] <= end_year)]

    # --- Build annual day totals ---
    annual_df = pd.DataFrame({'year': all_years})

    ce_days_annual = ce.groupby('year')['overlap_days'].sum().reset_index(name='ce_days')
    lf_days_annual = lf.groupby('year')['duration'].sum().reset_index(name='lf_days')
    hw_days_annual = hw.groupby('year')['duration'].sum().reset_index(name='hw_days')

    annual_df = annual_df.merge(ce_days_annual, on='year', how='left')
    annual_df = annual_df.merge(lf_days_annual, on='year', how='left')
    annual_df = annual_df.merge(hw_days_annual, on='year', how='left')
    annual_df = annual_df.fillna(0)

    # --- Day-based ratios per year ---
    # Conditioning event occurred (denominator > 0): ratio is well-defined, includes true zeros.
    # Conditioning event absent (denominator == 0): ratio is undefined -> NaN, excluded from mean.
    annual_df['heatwave_risk_given_lowflow'] = np.where(
        annual_df['lf_days'] > 0, annual_df['ce_days'] / annual_df['lf_days'], np.nan
    )
    annual_df['lowflow_risk_given_heatwave'] = np.where(
        annual_df['hw_days'] > 0, annual_df['ce_days'] / annual_df['hw_days'], np.nan
    )

    # Track how many years were undefined, for transparency/reporting
    n_undefined_hwrlf = annual_df['heatwave_risk_given_lowflow'].isna().sum()
    n_undefined_lfrhw = annual_df['lowflow_risk_given_heatwave'].isna().sum()

    if verbose:
        print(f"{'-'*70}")
        print(f"ANNUAL DAY TOTALS AND RATIOS ({start_year}-{end_year})")
        print(f"{'-'*70}")
        print(annual_df.to_string(index=False))
        print(f"\nUndefined (no conditioning-event days) years: "
              f"HWR_LF={n_undefined_hwrlf}/{N_years}, LFR_HW={n_undefined_lfrhw}/{N_years}")

    # --- Overall totals (sum across all years; pooled ratio, kept for comparison only) ---
    total_ce_days = annual_df['ce_days'].sum()
    total_lf_days = annual_df['lf_days'].sum()
    total_hw_days = annual_df['hw_days'].sum()

    heatwave_risk_given_lowflow = total_ce_days / total_lf_days if total_lf_days > 0 else np.nan
    lowflow_risk_given_heatwave = total_ce_days / total_hw_days if total_hw_days > 0 else np.nan

    if verbose:
        print(f"\n{'='*70}")
        print(f"OVERALL DAY-BASED RATIOS (SUM ACROSS {N_years} YEARS)")
        print(f"{'='*70}")
        print(f"Total CE days: {total_ce_days:.0f}")
        print(f"Total LF days: {total_lf_days:.0f}")
        print(f"Total HW days: {total_hw_days:.0f}")
        print()
        print(f"Heatwave risk given low flow (pooled) = {total_ce_days:.0f} / {total_lf_days:.0f} = "
              f"{heatwave_risk_given_lowflow:.4f}")
        print(f"Low-flow risk given a heatwave (pooled) = {total_ce_days:.0f} / {total_hw_days:.0f} = "
              f"{lowflow_risk_given_heatwave:.4f}")

    # --- Mean annual ratio, excluding undefined (NaN) years ---
    mean_annual_heatwave_risk = annual_df['heatwave_risk_given_lowflow'].mean(skipna=True)
    mean_annual_lowflow_risk = annual_df['lowflow_risk_given_heatwave'].mean(skipna=True)

    if verbose:
        print(f"\n{'='*70}")
        print(f"MEAN ANNUAL DAY-BASED RATIOS (NaN-EXCLUDED, OVER "
              f"{N_years - n_undefined_hwrlf}/{N_years} and {N_years - n_undefined_lfrhw}/{N_years} YEARS)")
        print(f"{'='*70}")
        print(f"Mean annual heatwave risk given low flow: {mean_annual_heatwave_risk:.4f}")
        print(f"Mean annual low-flow risk given a heatwave: {mean_annual_lowflow_risk:.4f}")

    return {
        'annual_df': annual_df,
        'heatwave_risk_given_lowflow_pooled': heatwave_risk_given_lowflow,
        'lowflow_risk_given_heatwave_pooled': lowflow_risk_given_heatwave,
        'mean_annual_heatwave_risk_given_lowflow': mean_annual_heatwave_risk,
        'mean_annual_lowflow_risk_given_heatwave': mean_annual_lowflow_risk,
        'n_undefined_years_hwrlf': n_undefined_hwrlf,
        'n_undefined_years_lfrhw': n_undefined_lfrhw,
    }


# --- Run for both periods ---
results_1991_2020 = compute_day_based_ce_ratios(
    compound_events, summer_low_flow_events, summer_heatwave_events,
    start_year=1991, end_year=2020, verbose=True
)

results_1961_1990 = compute_day_based_ce_ratios(
    compound_events, summer_low_flow_events, summer_heatwave_events,
    start_year=1961, end_year=1990, verbose=True
)

# --- Assemble into a summary table (using the mean-annual, NaN-excluded values) ---
risk_df = pd.DataFrame({
    '1961-1990': {
        'heatwave_risk_given_lowflow': results_1961_1990['mean_annual_heatwave_risk_given_lowflow'],
        'lowflow_risk_given_heatwave': results_1961_1990['mean_annual_lowflow_risk_given_heatwave'],
    },
    '1991-2020': {
        'heatwave_risk_given_lowflow': results_1991_2020['mean_annual_heatwave_risk_given_lowflow'],
        'lowflow_risk_given_heatwave': results_1991_2020['mean_annual_lowflow_risk_given_heatwave'],
    },
}).T

v1961 = risk_df.loc['1961-1990']
v1991 = risk_df.loc['1991-2020']
relative_delta = (v1991 - v1961) / v1961.replace(0, np.nan)
risk_df.loc['relative_delta'] = relative_delta

risk_df

----------------------------------------------------------------------
ANNUAL DAY TOTALS AND RATIOS (1991-2020)
----------------------------------------------------------------------
 year  ce_days  lf_days  hw_days  heatwave_risk_given_lowflow  lowflow_risk_given_heatwave
 1991      4.0     14.0     10.0                     0.285714                     0.400000
 1992      3.0     10.0     10.0                     0.300000                     0.300000
 1993      0.0      7.0      4.0                     0.000000                     0.000000
 1994      2.0      3.0      8.0                     0.666667                     0.250000
 1995      0.0      0.0      0.0                          NaN                          NaN
 1996      0.0      5.0      3.0                     0.000000                     0.000000
 1997      3.0     14.0      3.0                     0.214286                     1.000000
 1998      2.0      6.0      5.0                     0.333333                     0.40000

,heatwave_risk_given_lowflow,lowflow_risk_given_heatwave
1961-1990,0.247954,0.282424
1991-2020,0.402156,0.504291
relative_delta,0.621899,0.785581


## 6.4 Likelihood Multiplication Factor (LMF)

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
N_SHUFFLE = 5000
SEED = 42
VERBOSE = False   # set True to see per-step diagnostics

PERIODS = {
    "1961to1990": (1961, 1990),
    "1991to2020": (1991, 2020),
}

rng = np.random.default_rng(SEED)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Expand event catalogs into daily binary series (BM_LF, BM_HW)
# ─────────────────────────────────────────────────────────────────────────────

def events_to_daily_binary(events_df, start_year=1961, end_year=2020):
    """
    Expand an event catalog (start_date, duration) into a daily binary
    series (1 = day is inside a qualifying event, 0 = not), covering
    every day from start_year to end_year.
    """
    full_index = pd.date_range(f"{start_year}-01-01", f"{end_year}-12-31", freq="D")
    daily_flag = pd.Series(0, index=full_index, dtype=float)

    df = events_df.copy()
    df['start_date'] = pd.to_datetime(df['start_date'])

    for _, row in df.iterrows():
        event_dates = pd.date_range(row['start_date'], periods=int(row['duration']), freq='D')
        daily_flag.loc[daily_flag.index.isin(event_dates)] = 1

    return daily_flag


BM_LF_full = events_to_daily_binary(summer_low_flow_events, 1961, 2020)
BM_HW_full = events_to_daily_binary(summer_heatwave_events, 1961, 2020)

if VERBOSE:
    print(f"\n{'='*60}")
    print(f"LMF  —  lon={proto_lon}  lat={proto_lat}")
    print(f"{'='*60}")
    print("\n--- STEP 1: Expand event catalogs into daily binary series ---")
    print(f"Total BM_LF days (1961-2020): {BM_LF_full.sum():.0f}")
    print(f"Total BM_HW days (1961-2020): {BM_HW_full.sum():.0f}")

period_daily = {}
for period_label, (yr_s, yr_e) in PERIODS.items():
    period_daily[period_label] = {
        'BM_LF': BM_LF_full.loc[f"{yr_s}-01-01":f"{yr_e}-12-31"],
        'BM_HW': BM_HW_full.loc[f"{yr_s}-01-01":f"{yr_e}-12-31"],
    }

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Observed CE days (should match compound_events['overlap_days'].sum())
# ─────────────────────────────────────────────────────────────────────────────

def count_ce_days(bm_lf: pd.Series, bm_hw: pd.Series):
    overlap = (bm_lf == 1) & (bm_hw == 1)
    total = int(overlap.sum())
    per_year = overlap.groupby(overlap.index.year).sum().astype(int)
    return total, per_year


obs_results = {}
for period_label, (yr_s, yr_e) in PERIODS.items():
    bm_lf = period_daily[period_label]['BM_LF']
    bm_hw = period_daily[period_label]['BM_HW']
    ce_obs, ce_per_year = count_ce_days(bm_lf, bm_hw)
    obs_results[period_label] = {"ce_obs": ce_obs, "ce_per_year": ce_per_year}

    if VERBOSE:
        print(f"\n--- STEP 2: Observed CE days — {period_label} ({yr_s}-{yr_e}) ---")
        print(f"  Total CE days (observed) : {ce_obs}")
        print(f"  Mean CE days / year      : {ce_obs / (yr_e - yr_s + 1):.2f}")

# --- Sanity check against your compound_events catalog ---
compound_events['overlap_start'] = pd.to_datetime(compound_events['overlap_start'])
for period_label, (yr_s, yr_e) in PERIODS.items():
    catalog_sum = int(compound_events[
        (compound_events['overlap_start'].dt.year >= yr_s) &
        (compound_events['overlap_start'].dt.year <= yr_e)
    ]['overlap_days'].sum())
    reconstructed = obs_results[period_label]['ce_obs']
    if catalog_sum != reconstructed:
        print(f"⚠ WARNING [{period_label}]: catalog overlap_days sum ({catalog_sum}) "
              f"!= reconstructed daily CE count ({reconstructed}). "
              f"Check for overlapping events double-counted in the catalog, "
              f"or gaps in event date ranges.")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Null distribution via year-block shuffling
# ─────────────────────────────────────────────────────────────────────────────

def shuffle_ce_distribution(bm_lf, bm_hw, yr_s, yr_e, n_shuffle=1000, rng=None):
    """
    Shuffle BM_HW year-blocks (BM_LF held fixed) to build a null CE-day
    distribution. No re-streak-filtering is applied here since BM_LF/BM_HW
    are already derived from pre-filtered event catalogs (duration >= 3,
    summer window already enforced) rather than raw daily exceedances —
    shuffling whole year-blocks preserves each year's internal event
    structure exactly as it was in the catalog.
    """
    if rng is None:
        rng = np.random.default_rng()

    years = np.arange(yr_s, yr_e + 1)
    hw_blocks = [bm_hw[bm_hw.index.year == yr].values for yr in years]
    lf_arr = bm_lf.values
    ce_null = np.zeros(n_shuffle, dtype=int)

    for k in range(n_shuffle):
        idx_perm = rng.permutation(len(years))
        hw_shuffled = np.concatenate([hw_blocks[i] for i in idx_perm])
        ce_null[k] = int(np.nansum(lf_arr * hw_shuffled))

    return ce_null


shuffle_results = {}
for period_label, (yr_s, yr_e) in PERIODS.items():
    bm_lf = period_daily[period_label]['BM_LF']
    bm_hw = period_daily[period_label]['BM_HW']
    ce_null = shuffle_ce_distribution(bm_lf, bm_hw, yr_s, yr_e, n_shuffle=N_SHUFFLE, rng=rng)
    shuffle_results[period_label] = ce_null

    if VERBOSE:
        ce_obs = obs_results[period_label]["ce_obs"]
        print(f"\n--- STEP 3: Shuffle null distribution — {period_label} ---")
        print(f"  Shuffled mean CE  : {ce_null.mean():.2f}")
        print(f"  Shuffled 95th pct : {np.percentile(ce_null, 95):.1f}")
        print(f"  Observed CE       : {ce_obs}")
        print(f"  p-value           : {(ce_null >= ce_obs).mean():.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Compute LMF
# ─────────────────────────────────────────────────────────────────────────────
lmf_results = {}
for period_label in PERIODS:
    ce_obs = obs_results[period_label]["ce_obs"]
    ce_null = shuffle_results[period_label]
    ce_mean = ce_null.mean()
    ce_p95 = float(np.percentile(ce_null, 95))
    p_val = float((ce_null >= ce_obs).mean())
    lmf = (ce_obs / ce_mean) if ce_mean > 0 else np.nan
    sig = ce_obs > ce_p95

    lmf_results[period_label] = {
        "lmf": lmf, "ce_obs": ce_obs, "ce_mean": ce_mean,
        "ce_p95": ce_p95, "p_val": p_val, "sig": sig,
    }

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — LMF for both periods + relative delta
# ─────────────────────────────────────────────────────────────────────────────
lmf_1961 = lmf_results["1961to1990"]["lmf"]
lmf_1991 = lmf_results["1991to2020"]["lmf"]
ce_obs_1961 = lmf_results["1961to1990"]["ce_obs"]
ce_mean_1961 = lmf_results["1961to1990"]["ce_mean"]
ce_obs_1991 = lmf_results["1991to2020"]["ce_obs"]
ce_mean_1991 = lmf_results["1991to2020"]["ce_mean"]

print(f"LMF (1961-1990) = CE_obs / mean(CE_shuffled) = {ce_obs_1961} / {ce_mean_1961:.4f} = {lmf_1961:.4f}")
print(f"LMF (1991-2020) = CE_obs / mean(CE_shuffled) = {ce_obs_1991} / {ce_mean_1991:.4f} = {lmf_1991:.4f}")

if lmf_1961 != 0 and not np.isnan(lmf_1961):
    relative_delta_lmf = (lmf_1991 - lmf_1961) / lmf_1961
    print(f"Relative delta (LMF) = ({lmf_1991:.4f} - {lmf_1961:.4f}) / {lmf_1961:.4f} = {relative_delta_lmf:.4f}")
else:
    relative_delta_lmf = np.nan
    print("Relative delta (LMF): undefined (LMF for 1961-1990 is zero or NaN)")

lmf_df = pd.DataFrame({
    'period': ['1961-1990', '1991-2020'],
    'CE_obs': [ce_obs_1961, ce_obs_1991],
    'CE_shuffled_mean': [ce_mean_1961, ce_mean_1991],
    'LMF': [lmf_1961, lmf_1991],
}).set_index('period')
lmf_df.loc['relative_delta'] = (lmf_df.loc['1991-2020'] - lmf_df.loc['1961-1990']) / lmf_df.loc['1961-1990'].replace(0, np.nan)

lmf_df

LMF (1961-1990) = CE_obs / mean(CE_shuffled) = 55 / 9.6130 = 5.7214
LMF (1991-2020) = CE_obs / mean(CE_shuffled) = 179 / 37.6570 = 4.7534
Relative delta (LMF) = (4.7534 - 5.7214) / 5.7214 = -0.1692


,CE_obs,CE_shuffled_mean,LMF
period,,,
1961-1990,55.000000,9.613000,5.721419
1991-2020,179.000000,37.657000,4.753432
relative_delta,2.254545,2.917299,-0.169186
